# 7주차 실습: 문맥 기반 감정분석과 텍스트 임베딩

**핵심 질문: 문맥을 이해하는 감정분석은 기존 방식과 어떻게 다른가?**

NVIDIA 무료 개발용 API로 zero-shot 감정분류를 수행합니다. 예시 정답을 프롬프트에 넣지 않습니다. 사전·TF-IDF·BERT와 같은 문장을 비교합니다.

- 핵심 실습 약 45분: 준비 → 기준선 → NVIDIA 키 → 1건 점검 → 16건 분석 → 비교·저장
- 확장 실습 약 20분: Word2Vec, 문장 임베딩, BERT
- 모든 문장은 합성 교육용 자료입니다. 실제 댓글이나 개인정보를 보내지 않습니다.
- Colab: 파일 > Drive에 사본 저장. 위에서 아래로 셀 왼쪽 ▶를 누르세요. CPU로 가능합니다.
- 2026-10-15 확인: 모델 페이지의 Free Endpoint Available. 무료 개발용 제공 조건과 호출 제한은 바뀔 수 있습니다.

## 1. 준비 파일 다운로드
GitHub 파일을 Colab 작업 공간으로 내려받습니다. 기존 파일은 덮어쓰므로 사용자 원문은 `private_data/`에 별도로 보관하세요. 이 셀은 수업 제공 파일만 받습니다.

In [ ]:
from pathlib import Path
import urllib.request
import sys

BASE = "https://raw.githubusercontent.com/111usionBin/CNT3054/week07-nvidia-lab/week07/"
ROOT = Path("/content/CNT3054/week07") if "google.colab" in sys.modules else Path.cwd()
ROOT.mkdir(parents=True, exist_ok=True)
for name in ["lab.py", "requirements.txt", "data/train_reviews.csv", "data/context_reviews.csv"]:
    path = ROOT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(BASE + name, path)
sys.path.insert(0, str(ROOT))
print("준비 완료:", ROOT)

## 2. 패키지 설치
처음 실행은 다운로드 때문에 시간이 걸립니다. 빨간 오류가 없다면 기다리세요. 재시작을 요구하면 런타임을 재시작하고 1번부터 다시 실행합니다. NVIDIA API 자체에는 GPU가 필요하지 않습니다.

In [ ]:
%pip -q install "openai==2.3.0" "pandas>=2.2,<3" "scikit-learn>=1.6,<2"

## 3. 학습 자료와 진단 자료
6주차 합성 리뷰 120개로 기준선을 학습합니다. 새 진단 문장 16개는 학습에 넣지 않습니다. 앞 12개는 긍정/부정 각 6개이며, 나머지 4개는 중립/혼합의 범위 차이를 살펴봅니다. 교육용 라벨은 토론 가능한 기준이며 사람 전체의 정답을 대변하지 않습니다.

In [ ]:
import importlib
import lab
importlib.reload(lab)
import pandas as pd
from IPython.display import display
train, probe = lab.load_data()
print("학습:", len(train), "진단:", len(probe))
display(probe[["review_id", "phenomenon", "text", "target"]])

## 4. 사전과 TF-IDF 기준선
사전은 일부 긍정/부정 표현의 포함 여부를 합산합니다. 동점은 `-1=판정 보류`이며 중립의 정답이 아닙니다. TF-IDF+로지스틱 회귀는 6주차 합성 자료 전체에서 고정 설정으로 학습합니다. 진단 점수를 보고 설정을 고르면 더 이상 독립적인 평가가 아닙니다.

In [ ]:
result, tfidf_model = lab.run_baselines(train, probe)
display(result[["review_id", "text", "gold_label", "dictionary", "tfidf"]])

## 5. NVIDIA API 키 입력
[모델 페이지](https://build.nvidia.com/nvidia/nemotron-3-ultra-550b-a55b)에서 **Review Terms**를 읽고 본인이 동의한 뒤 **Generate API Key**를 누릅니다. 로그인/계정·전화 인증 등이 나오면 화면 안내를 따릅니다. 키를 발급받아 복사합니다.

아래 `RUN_NVIDIA = True`로 바꾼 뒤 실행하면 숨김 입력란이 나옵니다. 키를 붙여넣고 Enter를 누르세요. 코드 안에 키를 적거나 출력하지 않습니다.

기존 NVIDIA 키가 있으면 사용 권한을 확인해 재사용할 수 있습니다. 키 발급 화면이나 비밀키를 제출물에 넣지 마세요. 기본 False는 API를 호출하지 않습니다.

In [ ]:
from getpass import getpass
RUN_NVIDIA = False  # 준비되면 True로 변경
MODEL_ID = lab.MODEL_ID
client = None
if RUN_NVIDIA:
    api_key = getpass("NVIDIA API key (hidden): ")
    client = lab.make_client(api_key)
    del api_key
    print("클라이언트 준비 완료. 키는 출력하지 않습니다.")
else:
    print("API 미실행: RUN_NVIDIA=True로 바꾸고 이 셀부터 실행하세요.")

## 6. zero-shot 프롬프트 확인
분류 기준과 출력 형식만 제공하며 입력-정답 예시는 제공하지 않습니다. 원문은 분석 자료로 구분합니다. `gold_label`은 API로 전송하지 않습니다. `temperature=0`도 서버의 완전한 재현성을 보장하지는 않습니다.

In [ ]:
print(lab.PROMPT)

## 7. 문장 1개로 연결 점검
성공하면 `label`, `evidence`가 있는 원문 JSON과 파싱 결과가 나타납니다. 401은 키, 403은 권한·동의, 404는 모델 ID/제공 상태, 429는 요청 제한을 확인합니다. 자동 재시도는 429/서버 오류만 최대 2회이며, 모든 오류를 부정이나 중립으로 바꾸지 않습니다.

In [ ]:
if client is not None:
    smoke = lab.classify_one(client, "이 영화는 재미있지 않다.", model=MODEL_ID)
    display(pd.DataFrame([smoke]))
else:
    print("API 미실행. 실제 응답을 대신하는 예시 값은 만들지 않습니다.")

## 8. 같은 기준으로 16개 문장 분류
기본 실습은 점검 1회+문장 16회로 총 17개의 논리적 요청입니다. 오류 재시도 시 실제 호출 수는 증가합니다. 반복 실행 때마다 API가 다시 호출됩니다. 실패하면 먼저 오류를 해결한 후 재실행하세요.

In [ ]:
if client is not None:
    llm_results = lab.run_nvidia(client, probe, MODEL_ID)
    # 셀을 다시 실행해도 중복 열을 만들지 않습니다.
    base_cols = [c for c in result.columns if c not in llm_results.columns or c == "review_id"]
    result = result[base_cols].merge(llm_results, on="review_id", validate="one_to_one")
    display(result[["review_id", "gold_label", "llm_label", "evidence", "status"]])
else:
    print("NVIDIA 결과 열은 아직 없습니다. 아래 BERT 비교를 먼저 진행해도 됩니다.")

## 9. 확장 패키지 설치
Word2Vec·문장 임베딩·BERT에 필요합니다. Colab에는 보통 torch가 설치되어 있습니다. 처음에는 공개 모델 파일을 내려받습니다. 선택 실습을 건너뛰면 바로 13번으로 이동하세요.

In [ ]:
%pip -q install "torch>=2.6,<3" "transformers==4.57.1" "sentence-transformers==5.1.1" "gensim==4.4.0"

## 10. Word2Vec: 단어 하나를 숫자 32개로 표현
작은 합성 문장으로 Skip-gram을 학습합니다. 이 예제의 가까운 단어 순위는 신뢰할 만한 한국어 의미 사전이 아닙니다. 같은 단어는 어느 문장에서 나와도 하나의 고정 벡터를 사용합니다.

In [ ]:
w2v, neighbors = lab.word2vec_demo(train)
print("영화는 벡터의 모양:", w2v.wv["영화는"].shape)
print("앞 5개 값:", w2v.wv["영화는"][:5])
display(neighbors)

## 11. 문장 임베딩과 코사인 유사도
`paraphrase-multilingual-MiniLM-L12-v2`가 문장을 384차원 벡터로 바꿉니다. 정규화 후 내적을 계산하면 코사인 유사도입니다. 유사도가 높다고 감정이 같다는 뜻은 아닙니다. 부정문과 긍정문이 같은 주제를 공유하는지 확인하세요.

In [ ]:
sentences = ["이 영화는 재미있다.", "이 작품은 정말 즐겁다.", "이 영화는 재미있지 않다.", "공연은 일곱 시에 시작한다."]
embeddings, similarity = lab.sentence_similarity(sentences)
print("벡터 배열:", embeddings.shape)
display(pd.DataFrame(similarity, index=["S1","S2","S3","S4"], columns=["S1","S2","S3","S4"]).round(3))
for i, text in enumerate(sentences,1): print(f"S{i}: {text}")

## 12. BERT 감정분류 비교
NSMC로 미세조정된 다국어 BERT를 다운로드해 추론합니다. 이번 시간에는 BERT를 다시 학습하지 않습니다. 이 모델은 negative/positive만 출력하므로 중립/혼합을 직접 표현하지 못합니다. 처음 다운로드에는 시간이 걸립니다.

In [ ]:
bert_results, bert_revision = lab.run_bert(probe)
for col in bert_results.columns:
    result[col] = bert_results[col].to_numpy()
display(result[["review_id", "text", "gold_label", "bert", "bert_score"]])

## 13. 공통 12개 문장으로 비교
긍정/부정 문장에 한해 같은 분모로 계산합니다. 사전의 보류, LLM의 중립/혼합/형식 오류도 이 12개에서는 오답으로 남습니다. 이진 판정률과 전체 정확도·macro-F1을 함께 읽습니다. 모델의 학습 자료·규모가 달라 이 점수로 알고리즘 전체의 우열을 결론낼 수 없습니다.

In [ ]:
scores = lab.compare_metrics(result)
display(scores.round(3))
cols = [c for c in ["review_id", "text", "gold_label", "dictionary", "tfidf", "bert", "llm_label", "evidence", "status"] if c in result]
display(result[cols])
print("중립·혼합 문장은 위 지표에서 제외되며 아래에서 따로 확인합니다.")
display(result.loc[result.gold_label.isin(["중립", "혼합"]), cols])

## 14. 결과 저장과 다운로드
예측값, 지표, 오분류 검토표, 실행 조건을 새로운 폴더에 저장합니다. 키는 저장하지 않습니다. `raw_output`은 실제 API 응답이며, `evidence`는 검토할 인용이지 모델 내부 추론의 증거가 아닙니다.

In [ ]:
folder = lab.save_results(result, scores, model=MODEL_ID, extra={"nvidia_executed": "llm" in result, "bert_revision": globals().get("bert_revision")})
print("저장 위치:", folder)
import shutil
archive = shutil.make_archive(str(folder), "zip", folder)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("다운로드 파일:", archive)

## 15. 해석 기록
1. 사전과 LLM 또는 BERT가 다르게 예측한 문장 2개를 고르세요. 어떤 표현이 원인이라고 생각하나요?
2. 가장 유사한 문장 쌍과 감정 라벨이 같은 문장 쌍이 항상 일치하나요?
3. 중립·혼합 문장을 이진 분류로 강제하면 어떤 오류가 생기나요?
4. 여러분의 팀 자료에 적용한다면 분류 대상, 라벨 기준, 사람 검토 표본을 어떻게 정하겠습니까?

**제출 예시:** 결과 ZIP + 원문을 따옴표로 구분한 오류 분석 2개 + 적용 계획. 정답 라벨을 보고 프롬프트를 고친 결과는 탐색 결과이며 최종 평가가 아닙니다. 새로운 독립 평가 자료가 필요합니다.